# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhanish0711/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook documents the research question, provisional lane choice, decision framing, supporting dataset evidence, and boundary conditions for Assignment ML-02.

## 1. My lane (or freestyle) and why

**Chosen Lane:** **Lane 2 - Refresh / Content Opportunity Scoring**

**One-Paragraph Justification:**
For content strategists and SEO editorial teams deciding which high-traffic articles to review and update each week, we will build a ranked opportunity queue with risk probabilities and transparent reason codes from observable search and engagement signals, predicting content decline risk measured by Precision@50 and Average Precision. A wrong call costs editorial bandwidth spent rewriting healthy pages (False Positive) or unaddressed organic traffic decay on high-value pages (False Negative). A plain rule isn't enough because static heuristics (such as `days_since_last_update >= 180` and `impressions_90d >= 500`) miss complex non-linear interactions across position tiers, CTR benchmarks, and search intent. We will claim only decision-support recommendations evaluated on honest client-holdout validation.

In [1]:
# Confirming lane declaration
LANE_NAME = 'Lane 2 - Refresh / Content Opportunity Scoring'
LANE_TYPE = 'Core Lane'
print(f'Provisional Lane Selected: {LANE_NAME} ({LANE_TYPE})')


Provisional Lane Selected: Lane 2 - Refresh / Content Opportunity Scoring (Core Lane)


## 2. The question: decision, action, cost of a wrong call

### 1. What decision does this work improve?
This project improves the weekly **content review allocation decision**: out of hundreds or thousands of existing content items across client sites, which specific pages should an editorial team review, update, expand, or prune first to protect and recover organic search traffic?

### 2. Who acts on the output, and what do they do?
* **Who:** Content editors, SEO strategists, and copywriters.
* **Action:** Editors inspect the top-ranked pages from the review queue, evaluate the associated reason codes (e.g., `model_decline_risk`, `ctr_review_candidate`), update outdated facts/statistics, rewrite meta titles/descriptions for search intent alignment, or expand thin content sections.

### 3. What does a wrong answer cost?
* **Cost of a False Positive (flagging a healthy page):** Wasted editorial budget (~3-5 hours of writer/editor time per article, estimated at $150-$300 per review) and potential disruption of an already well-performing page.
* **Cost of a False Negative (missing a decaying page):** Continued organic traffic decay, loss of Page-1 SERP positions, reduced conversions, and a significantly higher cost to recover rankings once authority is lost.

### 4. Why does data or ML help at all?
Content decay is multi-dimensional. A single hand rule cannot distinguish between a page losing traffic due to intent mismatch, position decay, CTR collapse, or natural seasonal drift. A machine learning model can ingest 20+ non-linearly interacting search signals (impressions, positions, CTRs, freshness, engagement metrics) and compute a calibrated risk probability that ranks candidate pages far more accurately than fixed if-else thresholds.

In [2]:
# Framing Summary Checklist
framing = {
    'Decision': 'Weekly content refresh allocation',
    'User': 'SEO Editors & Content Strategists',
    'Action': 'Review, expand, update, or optimize meta tags',
    'FP Cost': 'Wasted editorial capacity ($150-$300/article)',
    'FN Cost': 'Unchecked traffic collapse and lost organic leads',
    'Eval Metric': 'Precision@50 & Average Precision (Client Holdout)'
}
for k, v in framing.items():
    print(f'{k:12s}: {v}')


Decision    : Weekly content refresh allocation
User        : SEO Editors & Content Strategists
Action      : Review, expand, update, or optimize meta tags
FP Cost     : Wasted editorial capacity ($150-$300/article)
FN Cost     : Unchecked traffic collapse and lost organic leads
Eval Metric : Precision@50 & Average Precision (Client Holdout)


## 3. Quick look at the data (2-3 real numbers)

Below we load the starter dataset (`data/raw/content_refresh_anonymized.csv`) and compute three empirical metrics that justify focusing on **Lane 2**:

In [3]:
import pandas as pd, numpy as np
from pathlib import Path

data_path = Path('data/raw/content_refresh_anonymized.csv')
df = pd.read_csv(data_path)

total_rows = len(df)
total_clients = df['client_id'].nunique()
down_rows = (df['trend_direction'] == 'down').sum()
down_pct = (down_rows / total_rows) * 100

high_demand = df[df['impressions_90d'] >= 500]
high_demand_count = len(high_demand)
high_demand_pct = (high_demand_count / total_rows) * 100
impression_share = (high_demand['impressions_90d'].sum() / df['impressions_90d'].sum()) * 100

high_demand_declining = (high_demand['trend_direction'] == 'down').sum()
high_demand_declining_pct = (high_demand_declining / high_demand_count) * 100

print('=' * 70)
print(f'1. Overall Inventory Decline Rate: {down_rows:,} / {total_rows:,} pages ({down_pct:.1f}%) in decline across {total_clients} clients.')
print(f'2. High-Demand Concentration: {high_demand_count:,} pages ({high_demand_pct:.1f}% of pages) capture {impression_share:.1f}% of total search impressions.')
print(f'3. High-Demand Decline Risk: {high_demand_declining:,} out of {high_demand_count:,} high-demand pages ({high_demand_declining_pct:.1f}%) are currently declining.')
print('=' * 70)

# Comparing baseline rule vs learned model Precision@50
import json
res_path = Path('outputs/model_results.json')
if res_path.exists():
    res = json.load(open(res_path))
    base_p50 = res['baseline']['baseline_precision_at_50']
    rf_p50 = res['models']['random_forest']['precision_at_50']
    print(f'4. Validation Precision@50 Improvement: Hand Rule = {base_p50:.3f} vs Random Forest = {rf_p50:.3f} ({rf_p50/base_p50:.2f}x gain).')


1. Overall Inventory Decline Rate: 16,262 / 30,000 pages (54.2%) in decline across 32 clients.
2. High-Demand Concentration: 16,726 pages (55.8% of pages) capture 99.0% of total search impressions.
3. High-Demand Decline Risk: 9,961 out of 16,726 high-demand pages (59.6%) are currently declining.
4. Validation Precision@50 Improvement: Hand Rule = 0.240 vs Random Forest = 0.680 (2.83x gain).


## 4. Careful words: what I can and can't claim

To maintain scientific integrity and avoid overselling results, we establish strict boundary conditions on our findings:

### What we CAN claim:
1. **Observed Associations:** We can report empirical correlations between observable search features (impressions, position tiers, CTR, content age) and content decline status.
2. **Decision-Support Prioritization:** We can demonstrate that an ML-based ranking model prioritizes true declining pages in the top 50 queue significantly better than a static hand-written rule.
3. **Client-Holdout Generalization:** We can evaluate how well our model generalizes to entirely unseen client sites using client-grouped validation splits.

### What we CANNOT claim:
1. **No Causal Proof:** We cannot claim that executing a refresh *causes* traffic recovery, as observational data without randomized controlled experiments cannot prove causality.
2. **No "Google Algorithm" Decoding:** We do not claim to have reverse-engineered search engine ranking algorithms or discovered universal ranking factors.
3. **No Automatic Guarantees:** We cannot promise guaranteed organic traffic growth from following model recommendations; the output is strictly human decision-support.

In [4]:
# Verification of careful language guidelines
claims = {
    'Valid': ['Observed correlation', 'Decision-support ranking', 'Precision@50 on holdout clients'],
    'Invalid': ['Causal proof of recovery', 'Decoding Google algorithm', 'Guaranteed ranking gains']
}
print('=== Claim Boundaries Verified ===')
for status, items in claims.items():
    print(f'{status} Claims: {', '.join(items)}')


=== Claim Boundaries Verified ===
Valid Claims: Observed correlation, Decision-support ranking, Precision@50 on holdout clients
Invalid Claims: Causal proof of recovery, Decoding Google algorithm, Guaranteed ranking gains


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.